# Case 4: Churn Detective - Telecom Retention Brief

This notebook turns the telecom churn dataset into a CMO-ready retention brief. The goal is not only to predict churn, but to explain who is at risk, why they are at risk, what offer to make, and how to measure the campaign in the first 60 days.

## Executive Summary

- Base churn rate: **36.16%**.
- Selected model for risk ranking: **Random Forest**.
- Held-out ROC-AUC: **0.7242**; PR-AUC: **0.5744**.
- Top-20% targeting captures **170 churners out of 280 targeted customers**.
- Top-20% precision: **60.71%**, or **1.68x lift** over random targeting.
- Main risk themes: short tenure, month-to-month contracts, repeated support calls, payment friction, electronic checks, and high monthly charges.
- Recommended plays: **Payment Friction Fix**, **Service Rescue**, and **Price-Lock / Contract Upgrade**.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'case4_telecom_churn.csv'
TABLES = ROOT / 'reports' / 'tables'
FIGURES = ROOT / 'reports' / 'figures'

df = pd.read_csv(DATA_PATH)
df.shape

## 1. Data Audit

The dataset has 7,000 customers, 20 input features, and one binary target: `churned`.

In [ ]:
pd.read_csv(TABLES / 'data_audit_summary.csv')

In [ ]:
Image(filename=str(FIGURES / 'base_churn_distribution.png'))

## 2. Churn Driver Evidence

The CMO asked for evidence, not only feature-importance bars. The tables and charts below show churn-rate gaps across business-readable groups.

In [ ]:
pd.read_csv(TABLES / 'churn_driver_summary.csv')

In [ ]:
display(Image(filename=str(FIGURES / 'churn_by_contract_type.png')))
display(Image(filename=str(FIGURES / 'churn_by_tenure_bucket.png')))
display(Image(filename=str(FIGURES / 'churn_by_support_calls.png')))

Key interpretation:

- Month-to-month customers churn far more than one-year and two-year customers.
- Early-tenure customers are fragile and need onboarding attention.
- Customers with repeated support calls are service-frustrated and should not receive only a generic discount.

In [ ]:
display(Image(filename=str(FIGURES / 'churn_by_late_payments.png')))
display(Image(filename=str(FIGURES / 'churn_by_payment_method.png')))
display(Image(filename=str(FIGURES / 'churn_by_monthly_charge_bucket.png')))

Payment friction and price pressure matter, but they should be targeted carefully. The strongest price play is not for every expensive customer; it is for high-risk customers with high monthly charges and flexible contracts.

## 3. Modeling

A Logistic Regression baseline gives interpretable directionality. A Random Forest slightly improves ranking metrics, so it is selected for churn-risk ranking.

In [ ]:
pd.read_csv(TABLES / 'model_candidate_comparison.csv')

In [ ]:
pd.read_csv(TABLES / 'baseline_top_coefficients.csv').head(12)

Model choice:

- **Random Forest** is used for ranking customers because it has the best ROC-AUC and PR-AUC.
- **Logistic Regression** remains useful for interpretation because coefficients show directionality.

## 4. Campaign Threshold

The CMO cannot target everyone. Instead of using a default 0.5 classification threshold, customers are ranked by churn probability and evaluated at campaign-size thresholds.

In [ ]:
pd.read_csv(TABLES / 'campaign_threshold_evaluation.csv')

Recommendation: start with the **top 20% highest-risk customers**. This gives a manageable outreach pool and captures one-third of churners with 1.68x lift over random targeting.

## 5. High-Risk Segments

Segments are designed for campaign action. Since customer risks can overlap, both primary segment and membership views are saved.

In [ ]:
pd.read_csv(TABLES / 'high_risk_segment_summary.csv')

In [ ]:
pd.read_csv(TABLES / 'high_risk_segment_membership_summary.csv')

In [ ]:
Image(filename=str(FIGURES / 'high_risk_primary_segments.png'))

## 6. Retention Plays

1. **Payment Friction Fix**: autopay incentive, billing reminders, and late-fee waiver after autopay enrollment.
2. **Service Rescue**: priority callback, issue owner, escalation SLA, and free tech-support trial.
3. **Price-Lock / Contract Upgrade**: price lock, annual-contract discount, or plan-rightsizing review.

The rough top-20% impact estimate is 17.0-25.5 saved customers and $7,648.98-$11,473.47 gross six-month revenue saved before offer costs, assuming 10-15% of identified churners are saved.

## 7. Limitations And Risks

- Observational data does not prove causality.
- The model predicts churn risk, not offer responsiveness.
- Discounts can waste margin on customers who would stay anyway.
- High-risk segments overlap and need campaign priority rules.
- Customer behavior may change next quarter.
- Features may proxy for sensitive or regulated attributes, so campaign policy should be reviewed.

## 8. 60-Day Measurement Plan

1. Rank eligible customers by churn probability.
2. Randomly split top-risk customers into treatment and holdout groups within each segment.
3. Apply segment-specific offers to treatment only.
4. Track 60-day churn, offer acceptance, support calls, late payments, and retained revenue.
5. Compare treatment versus holdout by segment.
6. Scale only plays with positive net value after offer costs.

Primary success metric: **60-day churn reduction in treatment group versus holdout group**.